In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# 1 Load data

## 1.1 Load population

In [2]:
pop_df = pd.read_csv('Pop ML V PIPE.csv', encoding='latin-1', sep='|')
pop_df.head(10)

,MSISDN,STATE_IN,SUBSCRIBER_TYPE_IN,RATE_PLAN,ACCOUNT_ACTIVATED_DATE
0,21697077044,ACTIVE,PREPAID,PRE - Classic,2003-12-19
1,21697076876,ACTIVE,PREPAID,PRE - Classic,2021-11-23
2,21697072547,ACTIVE,PREPAID,PRE - Classic,2020-09-14
3,21697068526,ACTIVE,PREPAID,PRE - Classic,2018-04-11
4,21697061045,ACTIVE,PREPAID,PRE - Classic,2003-11-15
5,21697060361,ACTIVE,PREPAID,PRE - Classic,2003-11-06
6,21697057138,ACTIVE,PREPAID,PRE - Classic,2004-02-01
7,21697046515,ACTIVE,PREPAID,PRE - Classic,2023-04-22
8,21697038087,SUSPENDED,PREPAID,PRE - Classic,2024-03-10
9,21697026893,SUSPENDED,PREPAID,PRE - Classic,2025-01-25


## 1.2 Load SOS

In [3]:
sos_df = pd.read_csv('DS_SOS_ML.csv', encoding='latin-1', sep='|')
sos_df.head(10)

,MSISDN,LOAN_ID,SOS_TYPE,CREDIT_DATE,CREDIT_AMOUNT,TOTAL_REIMBURSED,OUTSTANDING_AMOUNT,LAST_REIMBURSE_DATE,REIMBURSE_RATIO,REIMBURSE_STATUS,DAYS_SINCE_CREDIT
0,21650142555,528139435,CVAS_SOS_CR_DATA,09/01/2026,"4,5","4,5",0,23/01/2026,1,FULL,90
1,21693464197,528150918,CVAS_SOS_CR_SOLDE,09/01/2026,"0,5","0,5",0,09/01/2026,1,FULL,90
2,21697349073,528136747,CVAS_SOS_CR_SOLDE,09/01/2026,5,5,0,10/01/2026,1,FULL,90
3,21698641675,528163694,CVAS_SOS_CR_SOLDE,09/01/2026,3,3,0,15/01/2026,1,FULL,90
4,21694195742,528176161,CVAS_SOS_CR_DATA,09/01/2026,"0,9","0,9",0,09/01/2026,1,FULL,90
5,21693574965,528132046,CVAS_SOS_CR_SOLDE,09/01/2026,"2,5","2,5",0,19/01/2026,1,FULL,90
6,21696646805,528207682,CVAS_SOS_CR_SOLDE,09/01/2026,5,5,0,19/01/2026,1,FULL,90
7,21693370117,528155180,CVAS_SOS_CR_SOLDE,09/01/2026,"0,5","0,5",0,09/01/2026,1,FULL,90
8,21697385533,528223219,CVAS_SOS_CR_DATA,09/01/2026,"0,25","0,25",0,09/01/2026,1,FULL,90
9,21695867328,528161149,CVAS_SOS_CR_DATA,09/01/2026,"0,9","0,9",0,10/01/2026,1,FULL,90


# 2 Data Preparation for Model

## 2.1 Merge datasets on MSISDN

In [4]:
# Merge datasets on MSISDN
df = sos_df.merge(pop_df[['MSISDN', 'SUBSCRIBER_TYPE_IN', 'RATE_PLAN', 'ACCOUNT_ACTIVATED_DATE']], 
                   on='MSISDN', how='inner')

print(f"Merged dataset shape: {df.shape}")
print(f"\nColumns in merged dataset: {df.columns.tolist()}")
print(f"\nNull values after merge:")
print(df[['SOS_TYPE', 'SUBSCRIBER_TYPE_IN', 'RATE_PLAN', 'ACCOUNT_ACTIVATED_DATE', 'CREDIT_DATE', 'CREDIT_AMOUNT']].isnull().sum())

Merged dataset shape: (18562478, 14)

Columns in merged dataset: ['MSISDN', 'LOAN_ID', 'SOS_TYPE', 'CREDIT_DATE', 'CREDIT_AMOUNT', 'TOTAL_REIMBURSED', 'OUTSTANDING_AMOUNT', 'LAST_REIMBURSE_DATE', 'REIMBURSE_RATIO', 'REIMBURSE_STATUS', 'DAYS_SINCE_CREDIT', 'SUBSCRIBER_TYPE_IN', 'RATE_PLAN', 'ACCOUNT_ACTIVATED_DATE']

Null values after merge:
SOS_TYPE                      0
SUBSCRIBER_TYPE_IN        35004
RATE_PLAN                 35004
ACCOUNT_ACTIVATED_DATE    35004
CREDIT_DATE                   0
CREDIT_AMOUNT                 0
dtype: int64


## 2.2 Data Cleaning - Remove rows with null values in required features

In [5]:
# Data Cleaning - Remove rows with null values in required features
required_cols = ["MSISDN",'SOS_TYPE', 'SUBSCRIBER_TYPE_IN', 'ACCOUNT_ACTIVATED_DATE', 'CREDIT_DATE', 'CREDIT_AMOUNT']
df_clean = df.dropna(subset=required_cols).copy()

print(f"Dataset shape after removing nulls: {df_clean.shape}")
print(f"Rows removed: {df.shape[0] - df_clean.shape[0]}")
print(f"\nRemaining null values:")
print(df_clean[required_cols].isnull().sum())

Dataset shape after removing nulls: (18527474, 14)
Rows removed: 35004

Remaining null values:
MSISDN                    0
SOS_TYPE                  0
SUBSCRIBER_TYPE_IN        0
ACCOUNT_ACTIVATED_DATE    0
CREDIT_DATE               0
CREDIT_AMOUNT             0
dtype: int64


## 2.3 Convert date columns to datetime

In [6]:
# Convert date columns to datetime
df_clean['ACCOUNT_ACTIVATED_DATE'] = pd.to_datetime(df_clean['ACCOUNT_ACTIVATED_DATE'], errors='coerce')
df_clean['CREDIT_DATE'] = pd.to_datetime(df_clean['CREDIT_DATE'], errors='coerce')
# Convert amount from text like "4,5" to numeric 4.5
df_clean["CREDIT_AMOUNT"] = (
    df_clean["CREDIT_AMOUNT"]
    .astype(str)
    .str.replace(",", ".", regex=False)
)

df_clean["CREDIT_AMOUNT"] = pd.to_numeric(df_clean["CREDIT_AMOUNT"], errors="coerce")
# Remove rows where date conversion failed
df_clean = df_clean.dropna(subset=['ACCOUNT_ACTIVATED_DATE', 'CREDIT_DATE'])

print(f"Dataset shape after date conversion: {df_clean.shape}")
print(f"\nDate columns info:")
print(f"ACCOUNT_ACTIVATED_DATE - min: {df_clean['ACCOUNT_ACTIVATED_DATE'].min()}, max: {df_clean['ACCOUNT_ACTIVATED_DATE'].max()}")
print(f"CREDIT_DATE - min: {df_clean['CREDIT_DATE'].min()}, max: {df_clean['CREDIT_DATE'].max()}")

Dataset shape after date conversion: (7195753, 14)

Date columns info:
ACCOUNT_ACTIVATED_DATE - min: 2000-12-15 00:00:00, max: 2026-04-07 00:00:00
CREDIT_DATE - min: 2026-01-02 00:00:00, max: 2026-12-03 00:00:00


## 2.4 Feature Engineering - Derive new columns

In [7]:
# 4. prev_credit_gap: Days between previous and current credit for each customer
df_clean_sorted = df_clean.sort_values(['MSISDN', 'CREDIT_DATE']).copy()
df_clean_sorted['prev_credit_gap'] = df_clean_sorted.groupby('MSISDN')['CREDIT_DATE'].diff().dt.days

# Fill NaN for first credit of each customer with 0 (no previous credit)
df_clean_sorted['prev_credit_gap'] = df_clean_sorted['prev_credit_gap'].fillna(0)

# Update df_clean with the new feature
df_clean = df_clean_sorted.copy()

print("4. prev_credit_gap - Days since previous credit for each customer")
print(f"   Min: {df_clean['prev_credit_gap'].min():.0f}, Max: {df_clean['prev_credit_gap'].max():.0f}, Mean: {df_clean['prev_credit_gap'].mean():.2f}")
print(f"   Null values: {df_clean['prev_credit_gap'].isnull().sum()}")
print(f"   First-time customer records (gap=0): {(df_clean['prev_credit_gap'] == 0).sum()}")


4. prev_credit_gap - Days since previous credit for each customer
   Min: 0, Max: 335, Mean: 20.86
   Null values: 0
   First-time customer records (gap=0): 3088184


In [8]:
# 5. days_to_repay: Calculated from OUTSTANDING_AMOUNT and typical repayment period
# Using reference date to calculate days available for repayment
reference_date = df_clean['CREDIT_DATE'].max()
df_clean['days_to_repay'] = (reference_date - df_clean['CREDIT_DATE']).dt.days

print("5. days_to_repay - Days elapsed since credit was issued")
print(f"   Min: {df_clean['days_to_repay'].min()}, Max: {df_clean['days_to_repay'].max()}, Mean: {df_clean['days_to_repay'].mean():.2f}")
print(f"   Null values: {df_clean['days_to_repay'].isnull().sum()}")


5. days_to_repay - Days elapsed since credit was issued
   Min: 0, Max: 335, Mean: 168.15
   Null values: 0


In [9]:
# 6. is_lat: Flag for late payment (1 if OUTSTANDING_AMOUNT > 0, indicating unpaid balance; 0 if fully paid)
df_clean['is_lat'] = (df_clean['OUTSTANDING_AMOUNT'].astype(str).str.replace(",", ".", regex=False).astype(float) > 0).astype(int)

print("6. is_lat - Late payment indicator (1=Outstanding balance exists, 0=Paid)")
print(f"   Unique values: {df_clean['is_lat'].unique()}")
print(f"   Value counts:")
print(df_clean['is_lat'].value_counts())
print(f"   Percentage of late/outstanding: {(df_clean['is_lat'].sum() / len(df_clean) * 100):.2f}%")
print(f"   Null values: {df_clean['is_lat'].isnull().sum()}")


6. is_lat - Late payment indicator (1=Outstanding balance exists, 0=Paid)
   Unique values: [0 1]
   Value counts:
is_lat
0    5571640
1    1624113
Name: count, dtype: int64
   Percentage of late/outstanding: 22.57%
   Null values: 0


In [10]:
# Verify all new features are created
print("\n" + "=" * 80)
print("NEW FEATURES SUMMARY (prev_credit_gap, days_to_repay, is_lat)")
print("=" * 80)
print(f"\nShape after new features: {df_clean.shape}")
print(f"\nNew features statistics:")
print(df_clean[['prev_credit_gap', 'days_to_repay', 'is_lat']].describe())
print(f"\nSample data with new features:")
print(df_clean[['MSISDN', 'CREDIT_DATE', 'prev_credit_gap', 'days_to_repay', 'is_lat', 'OUTSTANDING_AMOUNT']].head(15))



NEW FEATURES SUMMARY (prev_credit_gap, days_to_repay, is_lat)

Shape after new features: (7195753, 17)

New features statistics:
       prev_credit_gap  days_to_repay        is_lat
count     7.195753e+06   7.195753e+06  7.195753e+06
mean      2.086442e+01   1.681501e+02  2.257044e-01
std       3.707734e+01   1.053434e+02  4.180454e-01
min       0.000000e+00   0.000000e+00  0.000000e+00
25%       0.000000e+00   6.300000e+01  0.000000e+00
50%       1.000000e+00   1.820000e+02  0.000000e+00
75%       3.000000e+01   2.740000e+02  0.000000e+00
max       3.350000e+02   3.350000e+02  1.000000e+00

Sample data with new features:
               MSISDN CREDIT_DATE  prev_credit_gap  days_to_repay  is_lat  \
877880    21620000528  2026-04-02              0.0            245       0   
1524135   21620000528  2026-05-02             30.0            215       0   
15246227  21620000528  2026-11-02            184.0             31       0   
2616562   21620001069  2026-11-02              0.0            

In [11]:
# 1. account_age_days: Calculate days between account activation and credit date
df_clean['account_age_days'] = (df_clean['CREDIT_DATE'] - df_clean['ACCOUNT_ACTIVATED_DATE']).dt.days

print("1. account_age_days - Days between account activation and credit date")
print(f"   Min: {df_clean['account_age_days'].min()}, Max: {df_clean['account_age_days'].max()}, Mean: {df_clean['account_age_days'].mean():.2f}")
print(f"   Null values: {df_clean['account_age_days'].isnull().sum()}")


1. account_age_days - Days between account activation and credit date
   Min: -95, Max: 9484, Mean: 2067.40
   Null values: 0


In [12]:
# 2. num_loans: Count number of loans per customer (MSISDN)
num_loans = df_clean.groupby('MSISDN').size().reset_index(name='num_loans')
df_clean = df_clean.merge(num_loans, on='MSISDN', how='left')

print("2. num_loans - Number of loans per customer")
print(f"   Min: {df_clean['num_loans'].min()}, Max: {df_clean['num_loans'].max()}, Mean: {df_clean['num_loans'].mean():.2f}")
print(f"   Null values: {df_clean['num_loans'].isnull().sum()}")
print(f"\nDistribution of num_loans:")
print(df_clean['num_loans'].value_counts().head(10))


2. num_loans - Number of loans per customer
   Min: 1, Max: 514, Mean: 26.76
   Null values: 0

Distribution of num_loans:
num_loans
5     262755
6     262686
4     259284
7     251832
3     250323
8     246024
9     237978
2     228324
10    225850
11    216898
Name: count, dtype: int64


In [13]:
# 3. avg_credit: Average credit amount per customer (MSISDN)
avg_credit = df_clean.groupby('MSISDN')['CREDIT_AMOUNT'].mean().reset_index(name='avg_credit')
df_clean = df_clean.merge(avg_credit, on='MSISDN', how='left')

print("3. avg_credit - Average credit amount per customer")
print(f"   Min: {df_clean['avg_credit'].min():.2f}, Max: {df_clean['avg_credit'].max():.2f}, Mean: {df_clean['avg_credit'].mean():.2f}")
print(f"   Null values: {df_clean['avg_credit'].isnull().sum()}")


3. avg_credit - Average credit amount per customer
   Min: 0.20, Max: 55.00, Mean: 1.56
   Null values: 0


In [14]:
# Verify all features are created
print("\n=== Feature Engineering Summary ===")
print(f"Shape after feature engineering: {df_clean.shape}")
print(f"\nNew features created:")
print(df_clean[['MSISDN', 'account_age_days', 'num_loans', 'avg_credit', 'CREDIT_AMOUNT']].head(10))
print(f"\nFeature statistics:")
print(df_clean[['account_age_days', 'num_loans', 'avg_credit']].describe())



=== Feature Engineering Summary ===
Shape after feature engineering: (7195753, 20)

New features created:
        MSISDN  account_age_days  num_loans  avg_credit  CREDIT_AMOUNT
0  21620000528              2599          3      3.0000           3.00
1  21620000528              2629          3      3.0000           2.00
2  21620000528              2813          3      3.0000           4.00
3  21620001069               991          1      2.0000           2.00
4  21620001196               214          4      1.6250           2.00
5  21620001196               456          4      1.6250           2.00
6  21620001196               486          4      1.6250           2.00
7  21620001196               486          4      1.6250           0.50
8  21620001696               888          4      0.4125           0.25
9  21620001696               888          4      0.4125           0.25

Feature statistics:
       account_age_days     num_loans    avg_credit
count      7.195753e+06  7.195753e+06  

## 2.5 Analyze cardinality and determine encoding strategy

In [15]:
# Create additional date-based features
from datetime import datetime

# Get the reference date (most recent date in dataset)
reference_date = df_clean['CREDIT_DATE'].max()

# DAYS_SINCE_CREDIT: Days since each credit (from reference date)
df_clean['DAYS_SINCE_CREDIT'] = (reference_date - df_clean['CREDIT_DATE']).dt.days

print("Created DAYS_SINCE_CREDIT feature:")
print(f"   Min: {df_clean['DAYS_SINCE_CREDIT'].min()}, Max: {df_clean['DAYS_SINCE_CREDIT'].max()}, Mean: {df_clean['DAYS_SINCE_CREDIT'].mean():.2f}")
print(f"   Null values: {df_clean['DAYS_SINCE_CREDIT'].isnull().sum()}")


Created DAYS_SINCE_CREDIT feature:
   Min: 0, Max: 335, Mean: 168.15
   Null values: 0


In [16]:
print("=" * 80)
print("CARDINALITY ANALYSIS & ENCODING STRATEGY")
print("=" * 80)

# Define column types - UPDATED with new features
categorical_cols = ['SOS_TYPE', 'SUBSCRIBER_TYPE_IN', 'RATE_PLAN', 'is_lat']
numeric_cols = ['CREDIT_AMOUNT', 'DAYS_SINCE_CREDIT', 'account_age_days', 'num_loans', 'avg_credit', 'prev_credit_gap', 'days_to_repay']

# Analyze categorical columns
print("\n📊 CATEGORICAL COLUMNS (for encoding):")
print("-" * 80)
encoder_strategy = {}
for col in categorical_cols:
    n_unique = df_clean[col].nunique()
    encoder_type = "OneHotEncoder" if n_unique <= 20 else "OrdinalEncoder"
    encoder_strategy[col] = {
        'unique_values': n_unique,
        'encoder': encoder_type,
        'values': df_clean[col].unique().tolist()[:10]  # Show first 10 values
    }
    print(f"\n{col}:")
    print(f"  Unique values: {n_unique}")
    print(f"  Recommended encoder: {encoder_type}")
    print(f"  Sample values: {df_clean[col].value_counts().head(5).to_dict()}")

# Analyze numeric columns
print("\n\n📈 NUMERIC COLUMNS (for scaling):")
print("-" * 80)
for col in numeric_cols:
    print(f"\n{col}:")
    print(f"  Min: {df_clean[col].min():.4f}")
    print(f"  Max: {df_clean[col].max():.4f}")
    print(f"  Mean: {df_clean[col].mean():.4f}")
    print(f"  Std: {df_clean[col].std():.4f}")
    print(f"  Null values: {df_clean[col].isnull().sum()}")


print(f"\n📋 DATASET SUMMARY:")
print(f"  Total records: {len(df_clean)}")
print(f"  Total features: {len(numeric_cols) + len(categorical_cols)}")
print(f"  Numeric features: {len(numeric_cols)}")
print(f"  Categorical features: {len(categorical_cols)}")

print("\n" + "=" * 80)
print("ENCODING RECOMMENDATION SUMMARY:")
print("=" * 80)
for col, strategy in encoder_strategy.items():
    print(f"  ✓ {col}: {strategy['unique_values']} values → {strategy['encoder']}")


CARDINALITY ANALYSIS & ENCODING STRATEGY

📊 CATEGORICAL COLUMNS (for encoding):
--------------------------------------------------------------------------------

SOS_TYPE:
  Unique values: 2
  Recommended encoder: OneHotEncoder
  Sample values: {'CVAS_SOS_CR_SOLDE': 3996617, 'CVAS_SOS_CR_DATA': 3199136}

SUBSCRIBER_TYPE_IN:
  Unique values: 3
  Recommended encoder: OneHotEncoder
  Sample values: {'PREPAID': 6616784, 'HYB': 578951, 'POSTPAID': 18}

RATE_PLAN:
  Unique values: 99
  Recommended encoder: OrdinalEncoder
  Sample values: {'PRE - Trankil TT': 2209244, 'PRE - 1=11': 1606035, 'PRE - TT 1500%': 480862, 'Hayya': 302880, 'PRE - TT 1000%': 293914}

is_lat:
  Unique values: 2
  Recommended encoder: OneHotEncoder
  Sample values: {0: 5571640, 1: 1624113}


📈 NUMERIC COLUMNS (for scaling):
--------------------------------------------------------------------------------

CREDIT_AMOUNT:
  Min: 0.2000
  Max: 55.0000
  Mean: 1.5561
  Std: 2.0565
  Null values: 0

DAYS_SINCE_CREDIT:
  Min:

## 2.6 Prepare features and target for modeling (BEFORE train/test split)

In [17]:
X = df_clean[  numeric_cols + categorical_cols].copy()
y = df_clean['OUTSTANDING_AMOUNT'].copy()

# Verify data integrity
print(f"Dataset size: {X.shape[0]} rows × {X.shape[1]} features")
print(f"\nNull values in features:")
print(X.isnull().sum())
print(f"\nFeature dtypes:")
print(X.dtypes)
print(f"\nNumeric features summary:")
print(X[numeric_cols].describe())


Dataset size: 7195753 rows × 11 features

Null values in features:
CREDIT_AMOUNT         0
DAYS_SINCE_CREDIT     0
account_age_days      0
num_loans             0
avg_credit            0
prev_credit_gap       0
days_to_repay         0
SOS_TYPE              0
SUBSCRIBER_TYPE_IN    0
RATE_PLAN             0
is_lat                0
dtype: int64

Feature dtypes:
CREDIT_AMOUNT         float64
DAYS_SINCE_CREDIT       int64
account_age_days        int64
num_loans               int64
avg_credit            float64
prev_credit_gap       float64
days_to_repay           int64
SOS_TYPE               object
SUBSCRIBER_TYPE_IN     object
RATE_PLAN              object
is_lat                  int32
dtype: object

Numeric features summary:
       CREDIT_AMOUNT  DAYS_SINCE_CREDIT  account_age_days     num_loans  \
count   7.195753e+06       7.195753e+06      7.195753e+06  7.195753e+06   
mean    1.556076e+00       1.681501e+02      2.067402e+03  2.675668e+01   
std     2.056486e+00       1.053434e+02    

# 3 Train-Test Split (80-20) - BEFORE fitting any transformers (prevent data leakage)


In [18]:
# Train-Test Split (80-20) - BEFORE fitting any transformers (prevent data leakage)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=50)

print(f"\n{'='*70}")
print(f"Train set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Train/Test ratio: {X_train.shape[0]/X_test.shape[0]:.1f}:1")
print(f"{'='*70}")


Train set: 5756602 samples
Test set: 1439151 samples
Train/Test ratio: 4.0:1


# XGBoost model

In [19]:
# Step 1: Setup preprocessing pipelines
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1,
        dtype=np.float32
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

print("✓ Preprocessing pipelines created")
print(f"  Numeric columns: {len(numeric_cols)}")
print(f"  Categorical columns: {len(categorical_cols)}")


✓ Preprocessing pipelines created
  Numeric columns: 7
  Categorical columns: 4


In [20]:
# Step 2: Fit preprocessing on train data ONLY (prevent data leakage)
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

# Apply log transformation to target (OUTSTANDING_AMOUNT)
# Convert target values from string to numeric before log transform
y_train = pd.to_numeric(
    y_train.astype(str).str.replace(",", ".", regex=False),
    errors="coerce"
)
y_test = pd.to_numeric(
    y_test.astype(str).str.replace(",", ".", regex=False),
    errors="coerce"
)

y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

print("✓ Data preprocessing completed")
print(f"  X_train prepared shape: {X_train_prepared.shape}")
print(f"  X_test prepared shape: {X_test_prepared.shape}")
print(f"\nTarget variable (OUTSTANDING_AMOUNT) - Original scale:")
print(f"  Train - Min: {y_train.min():.4f}, Max: {y_train.max():.4f}, Mean: {y_train.mean():.4f}")
print(f"  Test  - Min: {y_test.min():.4f}, Max: {y_test.max():.4f}, Mean: {y_test.mean():.4f}")
print(f"\nTarget variable - Log scale:")
print(f"  Train - Min: {y_train_log.min():.4f}, Max: {y_train_log.max():.4f}, Mean: {y_train_log.mean():.4f}")
print(f"  Test  - Min: {y_test_log.min():.4f}, Max: {y_test_log.max():.4f}, Mean: {y_test_log.mean():.4f}")


✓ Data preprocessing completed
  X_train prepared shape: (5756602, 11)
  X_test prepared shape: (1439151, 11)

Target variable (OUTSTANDING_AMOUNT) - Original scale:
  Train - Min: -0.0000, Max: 55.0000, Mean: 0.3663
  Test  - Min: -0.0000, Max: 55.0000, Mean: 0.3648

Target variable - Log scale:
  Train - Min: -0.0000, Max: 4.0254, Mean: 0.1779
  Test  - Min: -0.0000, Max: 4.0254, Mean: 0.1774


In [21]:
# Step 3: Train XGBoost model on log-transformed target
model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
    eval_metric="rmse"
)

print("\n" + "=" * 80)
print("TRAINING XGBOOST MODEL (for OUTSTANDING_AMOUNT prediction)")
print("=" * 80)

model.fit(X_train_prepared, y_train_log)
print("✓ Model training completed")

# Step 4: Make predictions
y_train_pred_log = model.predict(X_train_prepared)
y_test_pred_log = model.predict(X_test_prepared)

# Convert back to original scale
y_train_pred = np.expm1(y_train_pred_log)
y_test_pred = np.expm1(y_test_pred_log)

# Step 5: Evaluation metrics on original scale
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)

train_rmse = np.sqrt(train_mse)
test_rmse = np.sqrt(test_mse)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

train_mae = np.mean(np.abs(y_train - y_train_pred))
test_mae = np.mean(np.abs(y_test - y_test_pred))

train_mape = np.mean(np.abs((y_train - y_train_pred) / y_train)) * 100
test_mape = np.mean(np.abs((y_test - y_test_pred) / y_test)) * 100

print("\n" + "=" * 80)
print("XGBOOST REGRESSION MODEL RESULTS (OUTSTANDING_AMOUNT prediction)")
print("=" * 80)

print("\n📊 TRAINING SET METRICS:")
print(f"  MSE:  {train_mse:.4f}")
print(f"  RMSE: {train_rmse:.4f}")
print(f"  MAE:  {train_mae:.4f}")

print(f"  R²:   {train_r2:.4f}")

print("\n📊 TEST SET METRICS:")
print(f"  MSE:  {test_mse:.4f}")
print(f"  RMSE: {test_rmse:.4f}")
print(f"  MAE:  {test_mae:.4f}")

print(f"  R²:   {test_r2:.4f}")

print("\n" + "=" * 80)
print("MODEL PERFORMANCE SUMMARY")
print("=" * 80)
print(f"Generalization Gap (Test R² - Train R²): {test_r2 - train_r2:.4f}")
if test_r2 >= train_r2 - 0.05:
    print("✓ Good generalization - minimal overfitting")
else:
    print("⚠ Potential overfitting detected")

# Step 6: Feature importance analysis
try:
    feature_names = preprocessor.get_feature_names_out()
    print(f"\n📈 FEATURE IMPORTANCE ANALYSIS")
    print(f"Total features after preprocessing: {len(feature_names)}")
    
    if hasattr(model, "feature_importances_"):
        importances = model.feature_importances_
        top_indices = np.argsort(importances)[::-1][:15]
        
        print("\n🔝 Top 15 Feature Importances:")
        print("-" * 80)
        for rank, idx in enumerate(top_indices, 1):
            importance_pct = (importances[idx] / importances.sum()) * 100
            print(f"  {rank:2d}. {feature_names[idx]:30s} | {importances[idx]:.6f} ({importance_pct:.2f}%)")
    else:
        print("\n⚠ Feature importances not available for this model")
except Exception as e:
    print(f"\n⚠ Could not extract feature importances: {e}")

print("\n" + "=" * 80)



TRAINING XGBOOST MODEL (for OUTSTANDING_AMOUNT prediction)
✓ Model training completed

XGBOOST REGRESSION MODEL RESULTS (OUTSTANDING_AMOUNT prediction)

📊 TRAINING SET METRICS:
  MSE:  0.0790
  RMSE: 0.2810
  MAE:  0.0462
  R²:   0.9468

📊 TEST SET METRICS:
  MSE:  0.0858
  RMSE: 0.2929
  MAE:  0.0465
  R²:   0.9416

MODEL PERFORMANCE SUMMARY
Generalization Gap (Test R² - Train R²): -0.0052
✓ Good generalization - minimal overfitting

📈 FEATURE IMPORTANCE ANALYSIS
Total features after preprocessing: 11

🔝 Top 15 Feature Importances:
--------------------------------------------------------------------------------
   1. is_lat                         | 0.794859 (79.49%)
   2. CREDIT_AMOUNT                  | 0.131972 (13.20%)
   3. avg_credit                     | 0.022270 (2.23%)
   4. days_to_repay                  | 0.021772 (2.18%)
   5. DAYS_SINCE_CREDIT              | 0.011817 (1.18%)
   6. SOS_TYPE                       | 0.007950 (0.80%)
   7. SUBSCRIBER_TYPE_IN             | 0.